In [3]:
import numpy as np
import pandas as pd

In [4]:
data_claim = pd.read_csv('data/Data_Klaim.csv')
data_polis = pd.read_csv('data/Data_Polis.csv')

In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# >>> THE MAXIMUM STABILITY UPGRADE: LOG TRANSFORM <<<
weekly_df['Frequency_Log'] = np.log1p(weekly_df['Frequency'])
weekly_df['Total_Claim_Log'] = np.log1p(weekly_df['Total_Claim'])

# ==========================================
# 2. DLINEAR DECOMPOSITION (ON LOG DATA)
# ==========================================
print("2. Performing DLinear Decomposition on Log-Scaled Data...")

window_size = 4 

# We decompose the LOGGED features, not the raw billions
targets = ['Frequency_Log', 'Total_Claim_Log']

for col in targets:
    weekly_df[f'{col}_Trend'] = weekly_df[col].rolling(window=window_size).mean()
    weekly_df[f'{col}_Remain'] = weekly_df[col] - weekly_df[f'{col}_Trend']
    
    for lag in range(1, 5):
        weekly_df[f'{col}_Trend_Lag{lag}'] = weekly_df[f'{col}_Trend'].shift(lag)
        weekly_df[f'{col}_Remain_Lag{lag}'] = weekly_df[f'{col}_Remain'].shift(lag)

train_df = weekly_df.dropna().copy()

# ==========================================
# 3. TRAIN DLINEAR LINEAR LAYERS
# ==========================================
print("3. Training Dual Linear Layers in Log Space...")

model_params = {'alpha': 1.0}

# --- FREQUENCY MODELS ---
features_freq_trend = [f'Frequency_Log_Trend_Lag{i}' for i in range(1, 5)]
model_freq_trend = Ridge(**model_params)
model_freq_trend.fit(train_df[features_freq_trend], train_df['Frequency_Log_Trend'])

features_freq_remain = [f'Frequency_Log_Remain_Lag{i}' for i in range(1, 5)]
model_freq_remain = Ridge(**model_params)
model_freq_remain.fit(train_df[features_freq_remain], train_df['Frequency_Log_Remain'])

# --- TOTAL CLAIM MODELS ---
features_tot_trend = [f'Total_Claim_Log_Trend_Lag{i}' for i in range(1, 5)]
model_tot_trend = Ridge(**model_params)
model_tot_trend.fit(train_df[features_tot_trend], train_df['Total_Claim_Log_Trend'])

features_tot_remain = [f'Total_Claim_Log_Remain_Lag{i}' for i in range(1, 5)]
model_tot_remain = Ridge(**model_params)
model_tot_remain.fit(train_df[features_tot_remain], train_df['Total_Claim_Log_Remain'])

# ==========================================
# 4. RECURSIVE WEEKLY FORECASTING
# ==========================================
print("4. Forecasting Future Weeks recursively...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')
forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

current_history = weekly_df.copy()
weekly_predictions = []

for week_end in forecast_weeks:
    last_4 = current_history.tail(4)
    
    # --- FREQUENCY PREDICTION ---
    row_freq_trend = pd.DataFrame([{f'Frequency_Log_Trend_Lag{i}': last_4['Frequency_Log_Trend'].iloc[-i] for i in range(1, 5)}])
    row_freq_remain = pd.DataFrame([{f'Frequency_Log_Remain_Lag{i}': last_4['Frequency_Log_Remain'].iloc[-i] for i in range(1, 5)}])
    
    pred_f_trend_log = model_freq_trend.predict(row_freq_trend)[0]
    pred_f_remain_log = model_freq_remain.predict(row_freq_remain)[0]
    
    pred_freq_log = pred_f_trend_log + pred_f_remain_log
    pred_freq = np.expm1(pred_freq_log) # <--- CONVERT BACK TO REAL NUMBERS
    pred_freq = max(0, pred_freq) 
    
    # --- TOTAL CLAIM PREDICTION ---
    row_tot_trend = pd.DataFrame([{f'Total_Claim_Log_Trend_Lag{i}': last_4['Total_Claim_Log_Trend'].iloc[-i] for i in range(1, 5)}])
    row_tot_remain = pd.DataFrame([{f'Total_Claim_Log_Remain_Lag{i}': last_4['Total_Claim_Log_Remain'].iloc[-i] for i in range(1, 5)}])
    
    pred_t_trend_log = model_tot_trend.predict(row_tot_trend)[0]
    pred_t_remain_log = model_tot_remain.predict(row_tot_remain)[0]
    
    pred_total_log = pred_t_trend_log + pred_t_remain_log
    pred_total = np.expm1(pred_total_log) # <--- CONVERT BACK TO RUPIAH
    pred_total = max(0, pred_total)
    
    # Store Prediction
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })
    
    # Update History for the Recursive Loop
    new_row_data = {
        'Week_End_Date': week_end, 
        'Frequency': pred_freq, 'Total_Claim': pred_total,
        'Frequency_Log': pred_freq_log, 'Total_Claim_Log': pred_total_log
    }
    
    hist_plus_new = pd.concat([current_history, pd.DataFrame([new_row_data])], ignore_index=True)
    
    for col in targets:
        hist_plus_new[f'{col}_Trend'] = hist_plus_new[col].rolling(window=window_size).mean()
        hist_plus_new[f'{col}_Remain'] = hist_plus_new[col] - hist_plus_new[f'{col}_Trend']
        
    current_history = hist_plus_new.copy()

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 6. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_dlinear_logscaled.csv', index=False)

print("\n--- FINAL FORECAST (Log-Scaled DLinear Architecture) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_dlinear_logscaled.csv' saved! Safe from mathematical explosions!")

1. Processing Raw Data to Weekly Level...
2. Performing DLinear Decomposition on Log-Scaled Data...
3. Training Dual Linear Layers in Log Space...
4. Forecasting Future Weeks recursively...
5. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (Log-Scaled DLinear Architecture) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        218  4.871453e+07  1.061977e+10
1      2025-09        232  5.006399e+07  1.161485e+10
2      2025-10        241  5.052364e+07  1.217620e+10
3      2025-11        234  5.061158e+07  1.184311e+10
4      2025-12        242  5.065958e+07  1.225962e+10

File 'submission_dlinear_logscaled.csv' saved! Safe from mathematical explosions!
